# Scanning (precession) electron diffraction: centering the direct beam

This Jupyter Notebook was originally made for the "ESTEEM3 Workshop: Electron diffraction for solving engineering problems" at the Norwegian University of Science and Technology (NTNU) in Trondheim. It aimed to teach the participants how to center the direct beam in a Scanning Electron Diffraction (SED) dataset, as the diffraction pattern's center position moves over the field of view.

This is an important pre-processing step for several types of data processing, like azimuthal integration and template matching.

## Requirements

- HyperSpy 1.7.1
- pyXem 0.14.2

## Author

- 2022/06/22: Magnus Nord, TEM-Gemini, Department of Physics, NTNU. The Research Council of Norway is acknowledged for partially funding the creation of this Jupyter Notebook via the InCoMa project (Project number: 315475). Partial funding was also received from ESTEEM3 via the European Union's Horizon 2020 grant number 823717.

In [ ]:
%matplotlib qt

In [ ]:
import hyperspy.api as hs

## Beam centering

First, we load the dataset lazily. This dataset isn't really that large, only about 400 MB. But this processing is often necessary very large datasets, so here we use `lazy` processing as an example.

The dataset is nanocrystalline permalloy, which has been irradiated by Ga-ions. The dataset size has been reduced via cropping and rebinning, to make it quicker to download.

In [ ]:
s = hs.load("data/nanocrystalline_dscan.zspy", lazy=True)

Plotting it, we see a couple of interesting features:

- The direct beam moves about 20 pixels from the top left, to the bottom right of the dataset
- Some of the grains strongly diffract, leading to the diffraction disks being more intense than the direct beam

The latter is a potential problem, as the algorithm used to find the position of the direct beam might think one of the diffraction disks is the direct beam.

In [ ]:
s.plot()

There are many ways of finding the direct beam, from center of mass, cross correlation, to blurring the image and finding the most intense position.

Here, we'll use the "blurring" method.

In [ ]:
s_bs = s.get_direct_beam_position(method="blur", sigma=10, lazy_output=False)

This returns a `BeamShift` signal, with the same navigation shape as `s`, but with 1-signal dimension with 2 values. The first value is the x-position, and the second value is the y-position.

In [ ]:
s_bs

The dscan should be changing slowly and monotonically across the field-of-view, so we _should_ get a linear 2-D plane.

However, we clearly have a number of grains where the diffracted disks have been set as the central beam.

In [ ]:
s_bs.plot()

To visualize the x- and y-shifts in a more convenient way, change the signal to a DPC signal. This switches the navigation and signal dimensions.

In [ ]:
s_bs.to_dpcsignal().plot()

One way to fix this, is to fit a linear plane to both the x- and y-positions in `s_bs`.

`make_linear_plane` does exactly this, by replacing the content in `s_bs` with a fitted linear plane. This function also contains a `mask` parameter, so we could spend some time masking out the bad grains. However, these grains are randomly oriented, so in this specific example we don't need to use a mask. We can fit the plane to the whole dataset.

In [ ]:
s_bs.make_linear_plane()

Plotting this, we see that the beam shift is indeed a plane! However, we don't know if this is a good approximation for the direct beam position. To check this, we really have to correct the original signal `s`.

In [ ]:
s_bs.plot()

To do this, we use the `center_direct_beam` function, where we pass it the beam shifts we just calculated. By default this function works `inplace`, meaning it replaces the data in `s`.

In [ ]:
s.center_direct_beam(shifts=s_bs)

Running the `center_direct_beam` cell executes instantaneously, since the operation is done lazily. Ergo, the calculation hasn't actually been done yet.

To inspect the results, we have a couple of options:

- Run `compute()` to load the data into memory, and do the processing. This will be fine for this dataset, but for larger datasets it might crash your computer.
- Use `plot()` on the lazy `s` signal. This will do the calculations "on-demand", this means the plotting performance will be worse compared to the non-centered data.
- Saving the dataset to your hard drive, and then load this new, corrected, dataset lazily. This "trick" of saving intermediate results is useful when working with large datasets, since your hard drive have a lot more capacity compared to the available memory.

#### Plotting

Plotting is straight forward, as the only difference is that navigating across the dataset is slower.

In [ ]:
s.plot()

#### Saving the dataset

For big datasets, the centering procedure can take a long time. So making an intermediate dataset can be useful.

Note the chunk size here, which is the same size across all dimensions. This makes it easy to transpose the data, ergo viewing the diffraction patterns as a function of probe positions, and as probe positions as a function of diffraction pattern position.

In [ ]:
s.save("nanocrystalline_beam_centered.zspy", chunks=(32, 32, 32, 32))

Then we can reload the data

In [ ]:
s = hs.load("nanocrystalline_beam_centered.zspy", lazy=True)

#### Computing the data

Lastly, we load the data into memory, and do the computation. Note that if you work with larger datasets, this step might crash your computer.

In [ ]:
s.compute()

In [ ]:
s.plot()

## Using the centered data

Having put the direct beam in the center, lets do a common operation: azimuthal integration. This gives us the electron counts as a function of scattering angle.

In [ ]:
s.unit = "2th_deg"

In [ ]:
s.set_ai()

The `npt` parameter is the number of points in the radial direction.

In [ ]:
s_azi = s.get_azimuthal_integral1d(npt=80, lazy_output=False)

This gives the scattering distribution as a function probe position, which we see by plotting it.

In [ ]:
s_azi.plot()

The, perhaps, more interesting way of visualizing this, is transposing the data. Then we see the probe positions as a function of scattering angle, which is essentially a virtual annular dark field detector.

In [ ]:
s_azi.T.plot()